# Shor's Factoring Algorithm in PennyLane: Factoring $N=15$

The goal is to make the bridge between the lecture theory and an executable PennyLane implementation explicit:

1. reduce factoring to **order finding**;
2. encode modular multiplication as a quantum unitary;
3. use **Quantum Phase Estimation** to estimate a phase;
4. recover the order using continued fractions;
5. extract the non-trivial factors of $15$.

We use the standard small example:

$
N = 15, \qquad a = 2.
$

Since $\gcd(2,15)=1$, Shor's order-finding subroutine is applicable.

## 1. Mathematical background

Shor's factoring algorithm uses the following classical reduction.

Choose an integer $a$ with

$
1 < a < N, \qquad \gcd(a,N)=1.
$

The **order** of $a$ modulo $N$ is the smallest positive integer $r$ such that

$
a^r \equiv 1 \pmod N.
$

If $r$ is even and

$
a^{r/2} \not\equiv -1 \pmod N,
$

then the factors of $N$ can be obtained from

$
\gcd(a^{r/2}-1,N), \qquad \gcd(a^{r/2}+1,N).
$

For $N=15$ and $a=2$, the true order is $r=4$, because

$
2^4 = 16 \equiv 1 \pmod{15}.
$

Then

$
2^{r/2}=2^2=4,
$

and

$
\gcd(4-1,15)=3, \qquad \gcd(4+1,15)=5.
$

So the factorization is

$
15 = 3 \cdot 5.
$

In [22]:
import math
from fractions import Fraction
import numpy as np

try:
    import pennylane as qml
except ImportError as exc:
    raise ImportError(
        "PennyLane is not installed. In a fresh environment, run: pip install pennylane"
    ) from exc

np.set_printoptions(precision=4, suppress=True)

## 2. Classical sanity check: the order of 2 modulo 15

Before we build the quantum circuit, we verify the order classically. This is not part of the quantum speedup. It is just a sanity check so the notebook does not become mystical fog with syntax highlighting.

In [23]:
def classical_order(a: int, N: int) -> int:
    """Return the multiplicative order of a modulo N, assuming gcd(a, N)=1."""
    if math.gcd(a, N) != 1:
        raise ValueError("a and N must be coprime for order finding.")
    value = 1
    for r in range(1, N + 1):
        value = (value * a) % N
        if value == 1:
            return r
    raise RuntimeError("Order not found. Something is wrong with the assumptions.")

N = 15
a = 7
r_true = classical_order(a, N)
r_true

4

Expected output:

```text
4
```

## 3. The quantum unitary for modular multiplication

The quantum subroutine uses the unitary

$
U_a |x\rangle = |a x \bmod N\rangle.
$

For $N=15$, we need 4 target qubits because $2^4=16$. The computational basis states represent integers from $0$ to $15$.

For $x=0,1,\ldots,14$, we map

$
|x\rangle \mapsto |2x \bmod 15\rangle.
$

For the unused state $|15\rangle$, we leave it fixed. That keeps the operation a valid permutation on all 16 basis states.

In [24]:
def modular_multiplication_matrix(a_power: int, N: int, n_target: int) -> np.ndarray:
    """
    Construct the permutation matrix for |x> -> |a_power * x mod N>
    on n_target qubits. Basis states >= N are left unchanged.
    """
    dim = 2 ** n_target
    U = np.zeros((dim, dim), dtype=complex)

    for x in range(dim):
        if x < N:
            y = (a_power * x) % N
        else:
            y = x
        U[y, x] = 1.0

    return U

n_target = 4
U2 = modular_multiplication_matrix(a, N, n_target)

# Check unitarity: U dagger U should be identity.
np.allclose(U2.conj().T @ U2, np.eye(2 ** n_target))

True

Expected output:

```text
True
```

The matrix is unitary because it is a permutation matrix.

## 4. Phase estimation viewpoint

The lecture explains that Shor's algorithm relies on the same mechanism developed earlier:

- **phase kickback** writes eigenvalue information into a control register;
- **inverse QFT** decodes that phase pattern into computational-basis information;
- continued fractions turn the measured phase into a candidate order.

For order finding, the relevant unitary is $U_a$. Its eigenvalues have phases related to the period $r$:

$
U_a |\psi_s\rangle = e^{2\pi i s/r}|\psi_s\rangle, \qquad s=0,1,\ldots,r-1.
$

Quantum Phase Estimation estimates

$
\theta = \frac{s}{r}.
$

From the measured approximation to $\theta$, we recover $r$ using continued fractions.

## 5. Building the PennyLane QPE circuit

We use:

- 4 counting qubits;
- 4 target qubits;
- the target initialized to $|1\rangle$, because the sequence

$
1,2,4,8,1,2,4,8,\ldots
$

has period $4$ modulo $15$;
- controlled modular multiplication gates $U_a^{2^j}$;
- inverse QFT on the counting register;
- measurement of the counting register.

Using 4 counting qubits gives denominator $2^4=16$, enough to see the phases $0,1/4,1/2,3/4$ cleanly for this toy case.

In [30]:
n_count = 5
counting_wires = list(range(n_count))
target_wires = list(range(n_count, n_count + n_target))
all_wires = counting_wires + target_wires

dev = qml.device("default.qubit", wires=all_wires, shots=None)

# The controlled modular multiplication operation is the key component of the phase kickback stage.
# The funcion depends on the pennylane version
# def apply_controlled_modular_power(control_wire: int, power: int):
#     """Apply controlled U_a^(power), where U_a |x> = |a*x mod N>."""
#     a_power = pow(a, power, N)
#     U = modular_multiplication_matrix(a_power, N, n_target)
#     qml.ControlledQubitUnitary(U, control_wires=[control_wire], wires=target_wires)

def apply_controlled_modular_power(control_wire: int, power: int):
    """Apply controlled U_a^(power), where U_a |x> = |a*x mod N>."""
    a_power = pow(a, power, N)
    U = modular_multiplication_matrix(a_power, N, n_target)

    # New PennyLane API:
    # wires = [control wire] + target wires
    qml.ControlledQubitUnitary(
        U,
        wires=[control_wire] + target_wires
    )

@qml.qnode(dev)
def shor_qpe_distribution():
    # Prepare the target register in |1> = |0001>.
    qml.BasisState(np.array([0, 0, 0, 1]), wires=target_wires)

    # Create a uniform superposition on the counting register.
    for wire in counting_wires:
        qml.Hadamard(wires=wire)

    # Apply controlled powers U^(2^j).
    # This implements the phase kickback stage.
    for j, wire in enumerate(counting_wires):
        apply_controlled_modular_power(control_wire=wire, power=2 ** j)

    # Decode the phase with the inverse QFT.
    qml.adjoint(qml.QFT)(wires=counting_wires)

    # Return the exact probability distribution over the counting register.
    return qml.probs(wires=counting_wires)

probs = shor_qpe_distribution()
probs

array([0.25  , 0.2033, 0.1026, 0.0232, 0.    , 0.0088, 0.0127, 0.0049,
       0.    , 0.0033, 0.0057, 0.0025, 0.    , 0.0021, 0.0041, 0.002 ,
       0.    , 0.002 , 0.0041, 0.0021, 0.    , 0.0025, 0.0057, 0.0033,
       0.    , 0.0049, 0.0127, 0.0088, 0.    , 0.0232, 0.1026, 0.2033])

Depending on PennyLane's internal wire ordering convention, the visible peaks may appear in bit-reversed order. That is not a conceptual failure, just an indexing nuisance.

The important point is that the distribution should concentrate on values corresponding to phases compatible with denominator $4$.

In [26]:
def bitstring(x: int, width: int) -> str:
    return format(x, f"0{width}b")

# Display the most likely measurement outcomes.
peaks = sorted(
    [(idx, float(p)) for idx, p in enumerate(probs) if p > 1e-8],
    key=lambda item: item[1],
    reverse=True,
)

print("Most likely outcomes on the counting register:")
for idx, p in peaks:
    theta_est = idx / (2 ** n_count)
    print(f"  y = {idx:2d}  bitstring = {bitstring(idx, n_count)}  theta ≈ {theta_est:.4f}  probability = {p:.4f}")

Most likely outcomes on the counting register:
  y =  0  bitstring = 00000  theta ≈ 0.0000  probability = 0.2500
  y =  1  bitstring = 00001  theta ≈ 0.0312  probability = 0.2033
  y = 31  bitstring = 11111  theta ≈ 0.9688  probability = 0.2033
  y =  2  bitstring = 00010  theta ≈ 0.0625  probability = 0.1026
  y = 30  bitstring = 11110  theta ≈ 0.9375  probability = 0.1026
  y =  3  bitstring = 00011  theta ≈ 0.0938  probability = 0.0232
  y = 29  bitstring = 11101  theta ≈ 0.9062  probability = 0.0232
  y = 26  bitstring = 11010  theta ≈ 0.8125  probability = 0.0127
  y =  6  bitstring = 00110  theta ≈ 0.1875  probability = 0.0127
  y =  5  bitstring = 00101  theta ≈ 0.1562  probability = 0.0088
  y = 27  bitstring = 11011  theta ≈ 0.8438  probability = 0.0088
  y = 10  bitstring = 01010  theta ≈ 0.3125  probability = 0.0057
  y = 22  bitstring = 10110  theta ≈ 0.6875  probability = 0.0057
  y =  7  bitstring = 00111  theta ≈ 0.2188  probability = 0.0049
  y = 25  bitstring = 11001  

For the ideal order $r=4$, the clean phases are

$
0,\quad \frac14,\quad \frac12,\quad \frac34.
$

With 4 counting qubits, these correspond to

$
0,\quad 4,\quad 8,\quad 12
$

out of $16$. If your output is bit-reversed, the same information appears with the counting bits reversed.

## 6. Recovering the order from measurements

Given a measured integer $y$, the phase estimate is

$
\theta \approx \frac{y}{2^t},
$

where $t$ is the number of counting qubits.

We approximate $\theta$ by a rational number using continued fractions:

$
\theta \approx \frac{s}{r}.
$

The denominator is a candidate for the order. Sometimes it is a divisor of the true order or a useless value, so the classical post-processing must test candidates.

In [27]:
def candidate_denominator_from_measurement(y: int, n_count: int, max_denominator: int) -> int:
    """Use continued fractions to obtain a candidate denominator from y / 2^t."""
    if y == 0:
        return 0
    phase = Fraction(y, 2 ** n_count)
    return Fraction(phase).limit_denominator(max_denominator).denominator


def try_extract_factors_from_order_candidate(a: int, N: int, r_candidate: int):
    """
    Try r_candidate and its small multiples as possible orders.
    Return a tuple of non-trivial factors if successful, otherwise None.
    """
    if r_candidate <= 0:
        return None

    for multiplier in range(1, 2 * N + 1):
        r = multiplier * r_candidate

        # A valid order candidate should satisfy a^r = 1 mod N.
        if pow(a, r, N) != 1:
            continue

        if r % 2 != 0:
            continue

        x = pow(a, r // 2, N)
        if x == N - 1:
            continue

        p = math.gcd(x - 1, N)
        q = math.gcd(x + 1, N)

        if 1 < p < N and 1 < q < N:
            return tuple(sorted((p, q)))

    return None

print("Post-processing likely measurement outcomes:\n")
for y, p in peaks:
    denom = candidate_denominator_from_measurement(y, n_count, N)
    factors = try_extract_factors_from_order_candidate(a, N, denom)
    print(
        f"y={y:2d}, probability={p:.4f}, phase={y}/{2**n_count}, "
        f"candidate denominator={denom}, factors={factors}"
    )

Post-processing likely measurement outcomes:

y= 0, probability=0.2500, phase=0/32, candidate denominator=0, factors=None
y= 1, probability=0.2033, phase=1/32, candidate denominator=1, factors=(3, 5)
y=31, probability=0.2033, phase=31/32, candidate denominator=1, factors=(3, 5)
y= 2, probability=0.1026, phase=2/32, candidate denominator=15, factors=(3, 5)
y=30, probability=0.1026, phase=30/32, candidate denominator=15, factors=(3, 5)
y= 3, probability=0.0232, phase=3/32, candidate denominator=11, factors=(3, 5)
y=29, probability=0.0232, phase=29/32, candidate denominator=11, factors=(3, 5)
y=26, probability=0.0127, phase=26/32, candidate denominator=11, factors=(3, 5)
y= 6, probability=0.0127, phase=6/32, candidate denominator=11, factors=(3, 5)
y= 5, probability=0.0088, phase=5/32, candidate denominator=13, factors=(3, 5)
y=27, probability=0.0088, phase=27/32, candidate denominator=13, factors=(3, 5)
y=10, probability=0.0057, phase=10/32, candidate denominator=13, factors=(3, 5)
y=22,

The useful outputs recover the non-trivial factorization

$
15 = 3 \cdot 5.
$

The output $y=0$, corresponding to phase $0$, usually gives no useful denominator. That is normal. Shor's algorithm is probabilistic: not every run gives a useful sample, but useful samples occur with high enough probability that repetition fixes the issue.

## 7. A simple sampling-style wrapper

The previous circuit returned the exact distribution. A real quantum device would produce samples. The next cell simulates repeated measurements by sampling from the exact probability distribution.

In [28]:
rng = np.random.default_rng(seed=7)

samples = rng.choice(2 ** n_count, size=20, p=probs)
print("Simulated measurements:", samples.tolist())

for y in samples:
    denom = candidate_denominator_from_measurement(int(y), n_count, N)
    factors = try_extract_factors_from_order_candidate(a, N, denom)
    if factors is not None:
        print(f"Success from y={int(y)}: recovered factors {factors}")
        break
else:
    print("No useful sample found. Repeat the experiment.")

Simulated measurements: [17, 31, 30, 0, 1, 31, 0, 31, 31, 2, 1, 1, 1, 1, 2, 2, 31, 30, 14, 31]
Success from y=17: recovered factors (3, 5)


## 8. Summary: theory-to-practice map

| Lecture concept | Notebook implementation |
|---|---|
| Modular arithmetic | `modular_multiplication_matrix(a_power, N, n_target)` |
| Period/order $r$ | `classical_order(a, N)` for verification |
| Phase kickback | controlled applications of $U_a^{2^j}$ |
| Quantum Phase Estimation | counting register + controlled powers + inverse QFT |
| Inverse QFT | `qml.adjoint(qml.QFT)` |
| Continued fractions | `Fraction(...).limit_denominator(...)` |
| Factor extraction | gcd of $a^{r/2}\pm 1$ with $N$ |

This is the practical skeleton of Shor's algorithm. For real cryptographic numbers, the same conceptual pipeline remains, but modular exponentiation, error correction, and circuit depth become brutally expensive. The toy example is small because lectures need to end before civilization does.

## 9. Exercises

1. Change $a=2$ to $a=7$ and rerun the notebook. What is the order of $7$ modulo $15$?
2. Increase the number of counting qubits from 4 to 5. How does the phase resolution change?
3. Replace the exact distribution with a finite-shot PennyLane device and compare the sampling noise.
4. Explain why the unused computational basis state $|15\rangle$ must be handled carefully when representing arithmetic modulo $15$ with 4 qubits.
5. Identify precisely where phase kickback occurs in the circuit.

1. Still 4
2. Spacing becomes finer, denominator goes from 2^4 to 2^5

In [31]:
shots = 1000
dev_shots = qml.device("default.qubit", wires=all_wires, shots=shots)

@qml.qnode(dev_shots)
def shor_qpe_finite_shots():
    qml.BasisState(np.array([0, 0, 0, 1]), wires=target_wires)

    for wire in counting_wires:
        qml.Hadamard(wires=wire)

    for j, wire in enumerate(counting_wires):
        apply_controlled_modular_power(control_wire=wire, power=2 ** j)

    qml.adjoint(qml.QFT)(wires=counting_wires)

    return qml.probs(wires=counting_wires)

empirical_probs = shor_qpe_finite_shots()
empirical_probs

/home/pvela/university/qc/venv/lib/python3.12/site-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


array([0.252, 0.204, 0.111, 0.014, 0.   , 0.004, 0.008, 0.009, 0.   ,
       0.003, 0.005, 0.003, 0.   , 0.   , 0.002, 0.002, 0.   , 0.   ,
       0.006, 0.003, 0.   , 0.003, 0.009, 0.005, 0.   , 0.005, 0.015,
       0.003, 0.   , 0.013, 0.101, 0.22 ])

4. The unused state ∣15⟩ must be handled carefully because 4 qubits represent 16 states:
∣0⟩,…,∣15⟩

but arithmetic modulo 15 only uses:

∣0⟩,…,∣14⟩

5. Phase kickback occurs during the controlled modular multiplication gates:
for j, wire in enumerate(counting_wires):
    apply_controlled_modular_power(control_wire=wire, power=2 ** j)

Conceptually, this is where we apply:

∣j⟩∣ψs​⟩↦∣j⟩Ua2j​∣ψs​⟩

Ua​∣ψs​⟩=e2πis/r∣ψs​⟩

Ua2j​∣ψs​⟩=e2πi2js/r∣ψs​⟩